In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os
from numpy import sqrt
import matplotlib
from matplotlib import gridspec
from matplotlib.pyplot import *

from mpl_toolkits.axes_grid1 import make_axes_locatable

def latexify(fig_width=None, fig_height=None, columns=1):
    """Set up matplotlib's RC params for LaTeX plotting.
    Call this before plotting a figure.

    Parameters
    ----------
    fig_width : float, optional, inches
    fig_height : float,  optional, inches
    columns : {1, 2}
    """

    # code adapted from http://www.scipy.org/Cookbook/Matplotlib/LaTeX_Examples

    # Width and max height in inches for IEEE journals taken from
    # computer.org/cms/Computer.org/Journal%20templates/transactions_art_guide.pdf

    assert(columns in [1,2])

    if fig_width is None:
        fig_width = 3.39 if columns==1 else 6.9 # width in inches

    if fig_height is None:
        golden_mean = (sqrt(5)-1.0)/2.0    # Aesthetic ratio
        fig_height = fig_width*golden_mean # height in inches

    #MAX_HEIGHT_INCHES = 7.0
    #if fig_height > MAX_HEIGHT_INCHES:
    #    print("WARNING: fig_height too large:" + fig_height + 
    #          "so will reduce to" + MAX_HEIGHT_INCHES + "inches.")
    #    fig_height = MAX_HEIGHT_INCHES

    params = {'backend': 'ps',
              'text.latex.preamble': [r'\usepackage{gensymb}'],
              'axes.labelsize': 8, # fontsize for x and y labels (was 10)
              'axes.titlesize': 8,
              'font.size': 8, # was 10
              'legend.fontsize': 8, # was 10
              'xtick.labelsize': 8,
              'ytick.labelsize': 8,
              'text.usetex': True,
              'figure.figsize': [fig_width,fig_height],
              'font.family': 'serif',
              'figure.autolayout': True,
              'lines.linewidth' : 1.0
    }

    matplotlib.rcParams.update(params)


def format_axes(ax):
    
    SPINE_COLOR = 'gray'

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color(SPINE_COLOR)
        ax.spines[spine].set_linewidth(0.5)

    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')

    for axis in [ax.xaxis, ax.yaxis]:
        axis.set_tick_params(direction='out', color=SPINE_COLOR)

    return ax

In [3]:
import os
from matplotlib.pyplot import *
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec

def getBandstructure(dir1, nPlotBands, EShift):

    f = open(dir1,'r')
    
    
    i = 0
    j = 0
    l = 0
    
    k = []
    E = []
    
    kVec = []
    EVec = []
    
    counterPlotBands = 0
    
    for line in f:
        i = i+1
        a = line.split()
        if i==10:
            nbands = int(a[5])
            nk = int(a[7])
            print(nbands,nk)
        if i>15:
            if len(a)==0:
                j = j+0.5
                #print(j)
            else:
                if (int(j)==int(nbands/2-nPlotBands/2 + counterPlotBands) and int(j) < int(nbands/2+nPlotBands/2)):
                    if l<nk:
                        k.append(float(a[0]))
                        E.append(float(a[1])-EShift)
                        l = l+1
                    if l==nk:
                        kVec.append(k)
                        EVec.append(E)
                        l = 0
                        counterPlotBands += 1
                        k = []
                        E = []
                        
    return kVec, EVec

In [ ]:
emin = -1.0
emax = -0.4

nPlotBands = 50

TB_PGap_FullRelaxed = []
TB_PGap_FixBN = []
TB_PGap_Rigid = []
TB_PGap_Substrate = []

TB_SGap_FullRelaxed = []
TB_SGap_FixBN = []
TB_SGap_Rigid = []
TB_SGap_Substrate = []

TAPW_FullRelaxed = []
TAPW_FixBN = []
TAPW_Rigid = []
TAPW_Substrate = []

ang = []

#for Ang in ['0.0','0.1','0.3','0.396','0.5','0.558','0.769','0.915','1.381','1.530','2.060','2.558']:
for Ang in ['0.0','0.1','0.5','0.558','0.769','0.915','1.381','1.530','2.558']:
    ang.append(float(Ang))  
    for Type in ['FullRelaxed','FixBN','Rigid','Substrate']:        
        Temp = Ang
        Temp2 = Type
        
        EShift = -0.003
        dir1 = "Data_1219/Ang"+Temp+"/"+Temp2+"/TB/generate.bands.dat"
        kVec, EVec = getBandstructure(dir1, nPlotBands, EShift)
        i = 0
        for el1, el2 in zip(kVec,EVec):
            i = i+1
            if (i==nPlotBands/2):
                #ax0.plot(el1, el2,"r")
                valan1_max = max(el2)
            if (i==nPlotBands/2+1):
                #ax0.plot(el1, el2,"r")
                conduct1_min = min(el2)
            if (i==nPlotBands/2-1):
                #ax0.plot(el1, el2,"r")
                valance2_min = min(el2)
            if (i==nPlotBands/2-2):
                #ax0.plot(el1, el2,"r")
                valance3_max = max(el2)
                
        print('Primary Gap = ', (conduct1_min-valan1_max)*1000)
        print('Secondary Gap = ', (valance2_min-valance3_max)*1000)

        if Type == 'FullRelaxed':
            TB_PGap_FullRelaxed.append((conduct1_min-valan1_max)*1000)
            TB_SGap_FullRelaxed.append((valance2_min-valance3_max)*1000)
        elif Type == 'FixBN':
            TB_PGap_FixBN.append((conduct1_min-valan1_max)*1000)
            TB_SGap_FixBN.append((valance2_min-valance3_max)*1000)
        elif Type == 'Rigid':
            TB_PGap_Rigid.append((conduct1_min-valan1_max)*1000)
            TB_SGap_Rigid.append((valance2_min-valance3_max)*1000)
        elif Type == 'Substrate':
            TB_PGap_Substrate.append((conduct1_min-valan1_max)*1000)
            TB_SGap_Substrate.append((valance2_min-valance3_max)*1000)

        dir1 = "Data_1219/Ang"+Temp+"/"+Temp2+"/TAPW/generate.bands.dat"
        kVec, EVec = getBandstructure(dir1, 4, EShift)
        i = 0    
        for el1, el2 in zip(kVec,EVec):
            i = i+1
            if (i==2):
                #ax0.plot(el1, el2,".k")
                valan1_max = max(el2)
            if (i==3):
                #ax0.plot(el1, el2,".k")
                conduct1_min = min(el2)
                
        print('Primary Gap = ', (conduct1_min-valan1_max)*1000)
        
        if Type == 'FullRelaxed':
            TAPW_FullRelaxed.append((conduct1_min-valan1_max)*1000)
        elif Type == 'FixBN':
            TAPW_FixBN.append((conduct1_min-valan1_max)*1000)
        elif Type == 'Rigid':
            TAPW_Rigid.append((conduct1_min-valan1_max)*1000)
        elif Type == 'Substrate':
            TAPW_Substrate.append((conduct1_min-valan1_max)*1000)
        
        
        kpoint = 0.0
        gpoint = 0.035626
        mpoint = 0.056195
        gppoint = 0.092877
        kppoint = 0.097332

          
        os.chdir("./")

In [ ]:
# 创建图形
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Band Gap Comparison for Different Structural Models', 
             fontsize=16, fontweight='bold', y=1.02)

# 按照您的指定顺序：图1=Rigid, 图2=FixBN, 图3=Substrate, 图4=Full Relaxed
# 调整子图标题和数据对的顺序
titles = ['Rigid G/h-BN', 'Rigid Contacting BN layer', 'Rigid Remote NB layer', 'Suspended G/h-BN']
# 数据列表 - 按照新的顺序
data_pairs = [
    (TB_PGap_Rigid, TB_SGap_Rigid, TAPW_Rigid),
    (TB_PGap_FixBN, TB_SGap_FixBN, TAPW_FixBN),
    (TB_PGap_Substrate, TB_SGap_Substrate, TAPW_Substrate),
    (TB_PGap_FullRelaxed, TB_SGap_FullRelaxed, TAPW_FullRelaxed)
]

# 颜色和线型设置
colors = ['red', 'green', 'blue']
linestyles = ['-', '-', '--']
labels = ['Primary Gap', 'Secondary Gap', 'Average Mass']

# 绘制每个子图
for idx, ax in enumerate(axes.flat):
    pgap, sgap, tapw_pgap = data_pairs[idx]
    
    # 绘制三条曲线
    ax.plot(ang, pgap, color=colors[0], linestyle=linestyles[0], 
            linewidth=2, label=labels[0] if idx == 0 else "")
    ax.plot(ang, sgap, color=colors[1], linestyle=linestyles[1], 
            linewidth=2, label=labels[1] if idx == 0 else "")
    ax.plot(ang, tapw_pgap, color=colors[2], linestyle=linestyles[2], 
            linewidth=2, label=labels[2] if idx == 0 else "")
    
    # 设置子图属性
    ax.set_title(titles[idx], fontsize=12, fontweight='bold')
    ax.set_xlabel('Twist Angle ($\degree$)', fontsize=10)
    ax.set_ylabel('Band Gap (meV)', fontsize=10)
    
    # 统一设置y轴范围：[-10, 30]
    ax.set_ylim([-10, 30])
    
    # 设置x轴范围
    ax.set_xlim([min(ang), max(ang)])
    
    # 添加网格
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # 为每个子图添加小标题
    ax.text(0.05, 0.95, f'({chr(97+idx)})', transform=ax.transAxes, 
            fontsize=12, fontweight='bold', verticalalignment='top')

# 添加共用图例
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, 
           fontsize=11, framealpha=0.9, bbox_to_anchor=(0.5, 1.0))

# 调整布局
plt.tight_layout()

# 添加脚注
plt.figtext(0.5, 0.01, 
            'Note: TB = Tight Binding, TAPW = Truncated atomic plane wave', 
            ha='center', fontsize=10, style='italic', color='gray')

plt.show()

fig.savefig('band_gap_comparison.png', dpi=300, bbox_inches='tight', 
           facecolor='white', edgecolor='none', transparent=False)

In [ ]:
import numpy as np

data = np.column_stack((
    ang,
    TB_PGap_Rigid,      TB_SGap_Rigid,      TAPW_Rigid,
    TB_PGap_FixBN,      TB_SGap_FixBN,      TAPW_FixBN,
    TB_PGap_Substrate,  TB_SGap_Substrate,  TAPW_Substrate,
    TB_PGap_FullRelaxed,TB_SGap_FullRelaxed,TAPW_FullRelaxed
))

header = (
    "theta  "
    "TB_PGap_Rigid  TB_SGap_Rigid  TAPW_Rigid  "
    "TB_PGap_FixBN  TB_SGap_FixBN  TAPW_FixBN  "
    "TB_PGap_Substrate  TB_SGap_Substrate  TAPW_Substrate  "
    "TB_PGap_FullRelaxed  TB_SGap_FullRelaxed  TAPW_FullRelaxed"
)

np.savetxt(
    "gap_data_row1.dat",
    data,
    header=header,
    fmt="%.8f"
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['mathtext.fontset'] = 'stix'   # 让数学符号风格与 Times 匹配

data = np.loadtxt("gap_data_row2.dat")

theta = data[:, 0]

primary_gap1      = data[:, 1]
secondary_gap1    = data[:, 2]
averagemass_gap1  = data[:, 3]

primary_gap2      = data[:, 4]
secondary_gap2    = data[:, 5]
averagemass_gap2  = data[:, 6]

primary_gap3      = data[:, 7]
secondary_gap3    = data[:, 8]
averagemass_gap3  = data[:, 9]

primary_gap4      = data[:,10]
secondary_gap4    = data[:,11]
averagemass_gap4  = data[:,12]


# 设置子图
fig, axs = plt.subplots(1, 4, figsize=(20, 5))

# 子图1: Rigid G/h-BN
axs[0].plot(theta, averagemass_gap1, 'g-', marker='o', markersize=12, label='Secondary Gap')
axs[0].plot(theta, primary_gap1, 'r-', marker='*', markersize=20, label='Primary Gap')
axs[0].plot(theta, secondary_gap1, 'b-', marker='*', markersize=20, label='Secondary Gap')
axs[0].set_title('Rigid G/h-BN', fontsize=30)
axs[0].set_ylabel('Band Gap (meV)', fontsize=30)
axs[0].set_ylim(0,30)
axs[0].set_xlim(0,4)
#axs[0].legend(loc='best')
axs[0].tick_params(axis='both', which='major', labelsize=25)

# 子图2: Rigid Contacting BN layer
axs[1].plot(theta, secondary_gap2, 'b-', marker='*', markersize=20, label='Secondary Gap')
axs[1].plot(theta, primary_gap2, 'r-', marker='*', markersize=20, label='Primary Gap')
axs[1].plot(theta, averagemass_gap2, 'g-', marker='o', markersize=12, label='Secondary Gap')
axs[1].set_title('Rigid Contacting h-BN layer', fontsize=30)
axs[1].set_ylim(0,30)
axs[1].set_xlim(0,4)
#axs[1].legend(loc='best')
axs[1].tick_params(axis='y', labelleft=False)
axs[1].tick_params(axis='both', which='major', labelsize=25)

# 子图3: Rigid Remote NB layer
axs[2].plot(theta, secondary_gap3, 'b-', marker='*', markersize=20, label='Secondary Gap')
axs[2].plot(theta, primary_gap3, 'r-', marker='*', markersize=20, label='Primary Gap')
axs[2].plot(theta, averagemass_gap3, 'g-', marker='o', markersize=12, label='Secondary Gap')
axs[2].set_title('Rigid Remote h-NB layer', fontsize=30)
axs[2].set_ylim(0,30)
axs[2].set_xlim(0,4)
#axs[2].legend(loc='best')
axs[2].tick_params(axis='y', labelleft=False)
axs[2].tick_params(axis='both', which='major', labelsize=25)

# 子图4: Suspended G/h-BN
axs[3].plot(theta, secondary_gap4, 'b-', marker='*', markersize=20, label='Secondary Gap')
axs[3].plot(theta, primary_gap4, 'r-', marker='*', markersize=20, label='Primary Gap')
axs[3].plot(theta, averagemass_gap4, 'g-', marker='o', markersize=12, label='Secondary Gap')
axs[3].set_title('Suspended G/h-BN', fontsize=30)
axs[3].set_ylim(0,30)
axs[3].set_xlim(0,4)
#axs[3].legend(loc='best')
axs[3].tick_params(axis='y', labelleft=False)
axs[3].tick_params(axis='both', which='major', labelsize=25)

# 调整子图间的间隔
plt.tight_layout()
# 显示图形
plt.show()

fig.savefig('band_gap_comparison.png', dpi=300, bbox_inches='tight', 
           facecolor='white', edgecolor='none', transparent=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['mathtext.fontset'] = 'stix'   # 让数学符号风格与 Times 匹配

data = np.loadtxt("gap_data_row2.dat")

theta = data[:, 0]

primary_gap1      = data[:, 1]
secondary_gap1    = data[:, 2]
averagemass_gap1  = data[:, 3]

primary_gap2      = data[:, 4]
secondary_gap2    = data[:, 5]
averagemass_gap2  = data[:, 6]

primary_gap3      = data[:, 7]
secondary_gap3    = data[:, 8]
averagemass_gap3  = data[:, 9]

primary_gap4      = data[:,10]
secondary_gap4    = data[:,11]
averagemass_gap4  = data[:,12]

# 方法1：使用column_stack将多个数组按列组合
data2 = np.column_stack((theta, secondary_gap1, secondary_gap2, secondary_gap3, secondary_gap4))

# 或者方法2：使用vstack后转置（结果相同）
# data2 = np.vstack((theta, secondary_gap1, secondary_gap2, secondary_gap3, secondary_gap4)).T

# 保存为txt文件
np.savetxt("data2_output.txt", data2, 
           fmt='%.6f',  # 控制输出格式，保留6位小数
           delimiter='\t',  # 使用制表符分隔，也可以使用空格或逗号
           header='theta\tsecondary_gap1\tsecondary_gap2\tsecondary_gap3\tsecondary_gap4')  # 添加标题行

print(f"数据已保存为 'data2_output.txt'，形状为 {data2.shape}")

# 如果你想在控制台查看数据的前几行
print("\n数据前5行：")
print(data2[:5])